In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text

In [3]:
llm("What is your knowledge cutoff date?")

'My knowledge cutoff date is **2024-06**.'

In [4]:
#Without context, we get a general unspecific answer
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Yes—most of the time you can join after a course has started, but it depends on the course’s enrollment policy.

A few quick things to check:
- Whether late enrollment is allowed
- If there’s still space available
- Whether you’d be missing important prerequisites or early material
- Any deadline for adding the course

If you want, I can help you draft a short message to the instructor or course organizer asking if you can still join.


In [5]:
#Giving context:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [6]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [7]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [8]:
#RAG has two parts: retrieval and augmented generation
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [9]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

In [11]:
documents[1110]

{'id': 'f0ad880ffb',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 4: Analytics Engineering with dbt',
 'question': 'dbt + git: the main branch is "read-only" — how do I make changes?',
 'answer': 'dbt Cloud (and the dbt VS Code extension) protect the main/default branch by making it read-only. To make changes, create and switch to a feature branch:\n\nIn the dbt Cloud IDE, click "create new branch" in the top-left and give it a name.\n\nFrom the command line:\n\n```bash\ngit checkout -b your-feature-branch\n```\n\nMake your edits, commit, push, then open a pull request to merge back to main:\n\n```bash\ngit add .\ngit commit -m "your change description"\ngit push origin your-feature-branch\n```\n\nAfter the PR is merged on GitHub, the change appears on main.\n\nSee the dbt docs on [version control basics](https://docs.getdbt.com/docs/collaborate/git/version-control-basics) for more.'}

In [12]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [15]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are grea

In [16]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [17]:
search_results = search(question)

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [ ]:
search_results

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()